# Citizen-to-Policy Shortest-Path Analysis — Hypothesis 3

Extracts shortest paths between citizen-participation nodes and public/private policy hubs. These paths represent structural connectivity in the co-occurrence graph, not observed information flows.


In [ ]:
import pandas as pd
import networkx as nx

# 1. Load the preprocessed node and edge tables
nodes = pd.read_csv("nodes_with_type.csv")
edges = pd.read_csv("edges_filtered.csv")

# 2. Build the undirected graph
G = nx.from_pandas_edgelist(edges, "Source", "Target")

# 3. Extract citizen and policy-hub node IDs
citizen_ids = nodes[nodes["type"] == "citizen"]["Id"].tolist()
policy_ids = nodes[nodes["type"].isin(["policy", "private_policy"])]["Id"].tolist()

# 4. Compute shortest paths for valid citizen-policy pairs
valid_nodes = set(G.nodes)
paths = []
for citizen in citizen_ids:
    if citizen not in valid_nodes:
        continue
    for policy in policy_ids:
        if policy not in valid_nodes:
            continue
        try:
            paths.append(nx.shortest_path(G, source=citizen, target=policy))
        except nx.NetworkXNoPath:
            continue

# 5. Collect nodes and edges appearing on the extracted paths
path_nodes = set()
path_edges = []
for path in paths:
    path_nodes.update(path)
    path_edges.extend(zip(path[:-1], path[1:]))

# 6. Create path-specific node and edge tables
nodes_filtered = nodes[nodes["Id"].isin(path_nodes)]
edges_filtered = pd.DataFrame(path_edges, columns=["Source", "Target"])

# 7. Save outputs for downstream visualization and inspection
nodes_filtered.to_csv("shortest_path_nodes.csv", index=False)
edges_filtered.to_csv("shortest_path_edges.csv", index=False)
